In [2]:
%cd /Users/teresapaccosi/Desktop/VARIE/esperimenti_annotazioni/converted_file_last/
%ls

/Users/teresapaccosi/Desktop/VARIE/esperimenti_annotazioni/converted_file_last
altink_bio.tsv         dekker_bio.tsv
dejong_bio.tsv         westerheijden_bio.tsv


In [15]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from itertools import combinations
import krippendorff


COLUMNS = ['FOOD', 'DESCR', 'PR', 'CURE', 'INGR']
filepaths = ['altink_bio.tsv', 'dekker_bio.tsv', 'dejong_bio.tsv', 'westerheijden_bio.tsv']
NAMES = ['Annot1', 'Annot2', 'Annot3', 'Annot4']

def load_and_clean(path):
    df = pd.read_csv(path, sep="\t", dtype=str).fillna("O")
    df = df[COLUMNS]
    df = df.applymap(lambda x: re.sub(r'\\_', '_', str(x))) 
    return df

dfs = [load_and_clean(fp) for fp in filepaths]

if len(set(len(df) for df in dfs)) != 1:
    raise ValueError("Annotators have different lengths")


def compute_krippendorff(dfs):
    results = {}

    for col in COLUMNS:
        data = [df[col].tolist() for df in dfs]

        # FIX: safer union
        labels = set()
        for d in data:
            labels.update(d)

        if len(labels) < 2:
            results[col] = None
            continue

        alpha = krippendorff.alpha(
            reliability_data=np.array(data),
            level_of_measurement='nominal'
        )
        results[col] = alpha

    return results

alpha_per_col = compute_krippendorff(dfs)

print("\nKRIPPENDORFF col")
for k, v in alpha_per_col.items():
    print(f"{k}: {v if v is not None else 'N/A'}")


# def plot_confusion_matrices(dfs):
#     for (i, j) in combinations(range(len(dfs)), 2):
#         for col in COLUMNS:
#             a = dfs[i][col].tolist()
#             b = dfs[j][col].tolist()

#             labels = sorted(set(a + b))
#             cm = confusion_matrix(a, b, labels=labels)

#             plt.figure(figsize=(6,5))
#             sns.heatmap(cm, annot=True, fmt="d",
#                         xticklabels=labels,
#                         yticklabels=labels,
#                         cmap="Reds")
#             plt.title(f"{col}: {NAMES[i]} vs {NAMES[j]}")
#             plt.xlabel(NAMES[j])
#             plt.ylabel(NAMES[i])
#             plt.tight_layout()
#             plt.show()

# plot_confusion_matrices(dfs)

def bio_to_spans(tags):
    spans = []
    current = None
    start = None

    for i, t in enumerate(tags):
        t = t.split('|')[0]

        if t == 'O':
            if current:
                spans.append((start, i-1, current))
                current = None

        elif t.startswith('B-'):
            if current:
                spans.append((start, i-1, current))
            current = t[2:]
            start = i

        elif t.startswith('I-'):
            label = t[2:]
            if current != label:
                if current:
                    spans.append((start, i-1, current))
                current = label
                start = i

    if current:
        spans.append((start, len(tags)-1, current))

    return spans

def spans_to_sets(spans):
    return {(s, e, l) for s, e, l in spans}

def token_prf_with_krippendorff(dfs):
    # print("COL\tA\tB\tPRECISION\tRECALL\tF1\tKRIPP_ALPHA")
    print("COL\tA\tB\tKRIPP_ALPHA")

    for col in COLUMNS:
        data = [df[col].tolist() for df in dfs]

        for i, j in combinations(range(len(dfs)), 2):
            y_true = data[i]
            y_pred = data[j]

            labels = sorted(set(y_true + y_pred))

           
            # p, r, f1, _ = precision_recall_fscore_support(
            #     y_true,
            #     y_pred,
            #     labels=labels,
            #     average='macro',
            #     zero_division=0
            # )

           
            alpha = krippendorff.alpha(
                reliability_data=np.array([y_true, y_pred]),
                level_of_measurement='nominal'
            )

            print(
                f"{col}\t{NAMES[i]}\t{NAMES[j]}\t"
                f"{alpha:.3f}"
                # f"{p:.3f}\t{r:.3f}\t{f1:.3f}\t{alpha:.3f}"
            )

token_prf_with_krippendorff(dfs)

/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_33265/3569654776.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'\\_', '_', str(x)))
/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_33265/3569654776.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'\\_', '_', str(x)))
/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_33265/3569654776.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'\\_', '_', str(x)))
/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_33265/3569654776.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(r'\\_', '_', str(x)))



KRIPPENDORFF col
FOOD: 0.6172514094936826
DESCR: 0.3530317348867398
PR: 0.1720725915806416
CURE: 0.3848897846104691
INGR: 0.1822333937844941
COL	A	B	KRIPP_ALPHA
FOOD	Annot1	Annot2	0.545
FOOD	Annot1	Annot3	0.603
FOOD	Annot1	Annot4	0.568
FOOD	Annot2	Annot3	0.690
FOOD	Annot2	Annot4	0.625
FOOD	Annot3	Annot4	0.664
DESCR	Annot1	Annot2	0.149
DESCR	Annot1	Annot3	0.561
DESCR	Annot1	Annot4	0.211
DESCR	Annot2	Annot3	0.372
DESCR	Annot2	Annot4	0.439
DESCR	Annot3	Annot4	0.414
PR	Annot1	Annot2	-0.002
PR	Annot1	Annot3	-0.007
PR	Annot1	Annot4	-0.005
PR	Annot2	Annot3	0.055
PR	Annot2	Annot4	0.086
PR	Annot3	Annot4	0.509
CURE	Annot1	Annot2	0.302
CURE	Annot1	Annot3	0.425
CURE	Annot1	Annot4	0.391
CURE	Annot2	Annot3	0.307
CURE	Annot2	Annot4	0.399
CURE	Annot3	Annot4	0.480
INGR	Annot1	Annot2	0.234
INGR	Annot1	Annot3	0.058
INGR	Annot1	Annot4	0.100
INGR	Annot2	Annot3	0.056
INGR	Annot2	Annot4	0.023
INGR	Annot3	Annot4	0.432
